# Elasticsearch で日本語の全文検索をする (kuromoji)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_search/elasticsearch_ja.ipynb)

NLP クックブック 23-04 ／ 検索

自前の BM25 ではなく、検索エンジンの Elasticsearch で日本語の文書を全文検索したい。形態素解析器 kuromoji を組み込んだアナライザを作り、スコアやハイライト付きで検索結果を見たい。

In [1]:
# 所要時間の目安: Colab で 2〜4 分 (本体の配布物が約 840MB)
ES = "elasticsearch-9.5.6"
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/{ES}-linux-x86_64.tar.gz
!tar -xzf {ES}-linux-x86_64.tar.gz && rm {ES}-linux-x86_64.tar.gz
!{ES}/bin/elasticsearch-plugin install --batch analysis-kuromoji > /dev/null
# 学習用の設定: 1 台構成、手元 (127.0.0.1) だけで待ち受け、セキュリティ機能 (認証・TLS) は無効。
# 機械学習の機能は使わないので切る (メモリの節約)
!printf 'discovery.type: single-node\nnetwork.host: 127.0.0.1\nhttp.port: 9200\nxpack.security.enabled: false\nxpack.ml.enabled: false\n' >> {ES}/config/elasticsearch.yml
# ヒープ (Java のメモリ) は 1GB に固定する (指定しないと搭載メモリと役割から自動で決まる)
!printf -- '-Xms1g\n-Xmx1g\n' > {ES}/config/jvm.options.d/heap.options
# Elasticsearch は root では起動できないので、Colab に最初からいる daemon ユーザーの持ち物にしておく
!chown -R daemon:daemon {ES}
!pip install -q "elasticsearch>=9,<10" rank_bm25 fugashi unidic-lite

In [2]:
import subprocess, time, urllib.request
import elasticsearch

# バックグラウンドで起動する (Colab で 1 分ほど)。root では起動を拒否されるので daemon ユーザーで動かす
server = subprocess.Popen([f"{ES}/bin/elasticsearch"], user="daemon", group="daemon",
                          stdout=open("es.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):   # HTTP で応答するまで待つ
    if server.poll() is not None:   # 起動に失敗して終了した
        print(open("es.log").read()[-3000:]); break
    try:
        urllib.request.urlopen("http://localhost:9200", timeout=1); break
    except Exception:
        time.sleep(1)
es = elasticsearch.Elasticsearch("http://localhost:9200")
info = es.info()
print("サーバー", info["version"]["number"], "/ Lucene", info["version"]["lucene_version"], "/ クライアント", elasticsearch.__versionstr__)
print([p["component"] for p in es.cat.plugins(format="json")])

サーバー 9.5.6 / Lucene 10.5.2 / クライアント 9.5.1
['analysis-kuromoji']


In [3]:
# トークンの分かれ方を _analyze API で見る: 標準のアナライザ / kuromoji の通常モード / 検索モード
text = "関西国際空港で東京スカイツリーのお土産を買いました"
show = lambda **kw: print([t["token"] for t in es.indices.analyze(text=text, **kw)["tokens"]])
show(analyzer="standard")
show(tokenizer={"type": "kuromoji_tokenizer", "mode": "normal"})
show(tokenizer={"type": "kuromoji_tokenizer", "mode": "search"})
# 品詞での除外 (助詞・助動詞など) と原形化 (買い → 買う) まで入れたもの
show(tokenizer={"type": "kuromoji_tokenizer", "mode": "search"},
     filter=["kuromoji_baseform", "kuromoji_part_of_speech", "cjk_width", "ja_stop", "kuromoji_stemmer", "lowercase"])

['関', '西', '国', '際', '空', '港', 'で', '東', '京', 'スカイツリー', 'の', 'お', '土', '産', 'を', '買', 'い', 'ま', 'し', 'た']
['関西国際空港', 'で', '東京', 'スカイ', 'ツリー', 'の', 'お', '土産', 'を', '買い', 'まし', 'た']
['関西', '関西国際空港', '国際', '空港', 'で', '東京', 'スカイ', 'ツリー', 'の', 'お', '土産', 'を', '買い', 'まし', 'た']
['関西', '関西国際空港', '国際', '空港', '東京', 'スカイ', 'ツリー', '土産', '買う']


In [4]:
# 索引 (index) を作る。アナライザとマッピング (フィールドの型) はここで決める
settings = {
    "number_of_shards": 1, "number_of_replicas": 0,
    "analysis": {
        "tokenizer": {"ja_search": {"type": "kuromoji_tokenizer", "mode": "search"}},
        "analyzer": {"ja": {"type": "custom", "tokenizer": "ja_search",
                            "char_filter": ["kuromoji_iteration_mark"],
                            "filter": ["kuromoji_baseform", "kuromoji_part_of_speech", "cjk_width",
                                       "ja_stop", "kuromoji_stemmer", "lowercase"]}}},
    # BM25 のパラメータを変えた類似度も定義できる (b=0: 文書の長さで割り引かない)
    "similarity": {"bm25_nolen": {"type": "BM25", "k1": 1.2, "b": 0.0}},
}
mappings = {"properties": {
    "text": {"type": "text", "analyzer": "ja",
             "fields": {"nolen": {"type": "text", "analyzer": "ja", "similarity": "bm25_nolen"}}},
    "category": {"type": "keyword"},   # 絞り込み用 (解析しない)
}}
es.indices.delete(index="docs", ignore_unavailable=True)
es.indices.create(index="docs", settings=settings, mappings=mappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'docs'})

In [5]:
from elasticsearch import helpers

# この章の「BM25 で日本語のキーワード検索をする」と同じ文書
docs = [
    ("地理", "富士山は標高 3776 メートルで、日本で最も高い山である。"),
    ("建築", "東京スカイツリーは高さ 634 メートルの電波塔である。"),
    ("地理", "琵琶湖は滋賀県にある日本最大の湖である。"),
    ("地理", "信濃川は全長 367 キロメートルで、日本で最も長い川である。"),
    ("技術", "形態素解析とは、文を単語に分割して品詞を付与する処理である。"),
    ("技術", "BERT は Transformer のエンコーダを用いた事前学習モデルである。"),
]
actions = ({"_index": "docs", "_id": str(i), "_source": {"category": c, "text": t}} for i, (c, t) in enumerate(docs))
print(helpers.bulk(es, actions, refresh=True))   # refresh=True: 登録した文書をすぐ検索できるようにする
print(es.count(index="docs")["count"], "件")

(6, [])
6 件


In [6]:
ja = lambda s: [t["token"] for t in es.indices.analyze(index="docs", analyzer="ja", text=s)["tokens"]]

def search(query, field="text", category=None, size=2):
    q = {"match": {field: query}}   # クエリも同じアナライザで解析され、どれかの語を含む文書が当たる (OR)
    if category:   # bool の filter はスコアに影響しない絞り込み
        q = {"bool": {"must": [q], "filter": [{"term": {"category": category}}]}}
    res = es.search(index="docs", query=q, size=size, highlight={"fields": {field: {}}})
    print(query, ja(query), f"→ {res['hits']['total']['value']} 件ヒット")
    for h in res["hits"]["hits"]:
        print(f"  {h['_score']:.3f}  [{h['_source']['category']}] {h['highlight'][field][0]}")

for q in ["日本で一番高い山", "BERT とは何ですか", "日本一大きい湖はどこ？"]:
    search(q)
search("日本", size=4)
search("日本", category="技術")
print(ja("高さ"), ja("お土産を買いました"))

日本で一番高い山 ['日本', '一番', '高い', '山'] → 4 件ヒット
  3.235  [地理] 富士山は標高 3776 メートルで、<em>日本</em>で最も<em>高い</em><em>山</em>である。
  1.021  [建築] 東京スカイツリーは<em>高</em>さ 634 メートルの電波塔である。
BERT とは何ですか ['bert', '何'] → 1 件ヒット
  1.611  [技術] <em>BERT</em> は Transformer のエンコーダを用いた事前学習モデルである。
日本一大きい湖はどこ？ ['日本一', '大きい', '湖', 'どこ'] → 1 件ヒット
  1.704  [地理] 琵琶湖は滋賀県にある日本最大の<em>湖</em>である。
日本 ['日本'] → 3 件ヒット
  0.767  [地理] 琵琶湖は滋賀県にある<em>日本</em>最大の湖である。
  0.687  [地理] 富士山は標高 3776 メートルで、<em>日本</em>で最も高い山である。
  0.653  [地理] 信濃川は全長 367 キロメートルで、<em>日本</em>で最も長い川である。
日本 ['日本'] → 0 件ヒット
['高い'] ['土産', '買う']


In [7]:
# match は既定で OR。minimum_should_match で「クエリの語のうち何割を含むか」を指定できる
for msm in ["1", "75%", "100%"]:
    res = es.search(index="docs", query={"match": {"text": {"query": "日本で一番高い山", "minimum_should_match": msm}}})
    print(f"{msm:>4}", [(h["_id"], round(h["_score"], 3)) for h in res["hits"]["hits"]])

   1 [('0', 3.235), ('1', 1.021), ('2', 0.767), ('3', 0.653)]
 75% [('0', 3.235)]
100% []


In [8]:
# 自前の BM25 (この章の「BM25 で日本語のキーワード検索をする」の rank_bm25 + fugashi) とスコアを並べる
import numpy as np
from fugashi import Tagger
from rank_bm25 import BM25Okapi

tagger = Tagger()
tok = lambda s: [w.feature.orthBase or w.surface for w in tagger(s.lower())
                 if w.feature.pos1 not in ("助詞", "助動詞", "補助記号", "空白")]
bm25 = BM25Okapi([tok(t) for _, t in docs])
for q in ["日本で一番高い山", "日本 最も 長い", "メートル"]:
    es_score = np.zeros(len(docs))
    for h in es.search(index="docs", query={"match": {"text": q}})["hits"]["hits"]:
        es_score[int(h["_id"])] = h["_score"]
    print(q, "\n   Elasticsearch:", es_score.round(2).tolist(), "\n   rank_bm25    :", bm25.get_scores(tok(q)).round(2).tolist())

日本で一番高い山 
   Elasticsearch: [3.24, 1.02, 0.77, 0.65, 0.0, 0.0] 
   rank_bm25    : [2.41, 0.58, 0.0, 0.0, 0.0, 0.0]
日本 最も 長い 
   Elasticsearch: [1.71, 0.0, 0.77, 3.08, 0.0, 0.0] 
   rank_bm25    : [0.58, 0.0, 0.0, 1.95, 0.0, 0.0]
メートル 
   Elasticsearch: [1.02, 1.02, 0.0, 0.0, 0.0, 0.0] 
   rank_bm25    : [0.58, 0.58, 0.0, 0.0, 0.0, 0.0]


In [9]:
# スコアの内訳 (explain): Lucene の BM25 の式の各項が見える
def print_tree(e, depth=0):
    print("  " * depth + f"{e['value']:.3f}  {e['description'][:60]}")
    for d in e["details"]:
        print_tree(d, depth + 1)

res = es.search(index="docs", query={"match": {"text": "日本"}}, explain=True, size=1)
print_tree(res["hits"]["hits"][0]["_explanation"])

0.767  weight(text:日本 in 2) [PerFieldSimilarity], result of:
  0.767  score(freq=1.0), computed as boost * idf * tf from:
    2.200  boost
    0.693  idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
      3.000  n, number of documents containing term
      6.000  N, total number of documents with field
    0.503  tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)
      1.000  freq, occurrences of term within document
      1.200  k1, term saturation parameter
      0.750  b, length normalization parameter
      6.000  dl, length of field
      7.833  avgdl, average length of field


In [10]:
# BM25 の b を 0 にしたフィールド (text.nolen) では、文書の長さで割り引かなくなる
for field in ["text", "text.nolen"]:
    hits = es.search(index="docs", query={"match": {field: "日本"}})["hits"]["hits"]
    print(f"{field:10}", [(h["_id"], len(ja(h["_source"]["text"])), round(h["_score"], 3)) for h in hits])

text       [('2', 6, 0.767), ('0', 8, 0.687), ('3', 10, 0.653)]
text.nolen [('0', 8, 0.693), ('2', 6, 0.693), ('3', 10, 0.693)]


In [11]:
# 後片付け: サーバーを止める (終了処理に 30 秒ほどかかる)
es.close()
server.terminate()
server.wait(timeout=120)
print("終了コード", server.returncode)   # 143 = SIGTERM で終了した

終了コード 143


- Elasticsearch は、検索ライブラリ Lucene の転置索引を HTTP の API で使えるようにした検索サーバー。この版 (9.5.6) は Lucene 10.5.2 を使っている。スコアの既定は BM25 で、explain の内訳のとおり k1=1.2、b=0.75、IDF は log(1 + (N − n + 0.5) / (n + 0.5))。boost の 2.2 は k1 + 1 にあたり、付録「仕組みを理解するための実装」の「BM25 を自分で実装してパラメータの働きを確かめる」の式と同じ形になる。
- 自前の BM25（rank_bm25 の BM25Okapi）と比べると、スコアが 0 より大きい文書の順位は同じだが、スコアの値とヒットする文書の数が違う。BM25Okapi の IDF は log((N − n + 0.5) / (n + 0.5)) なので、6 件中 3 件に出る「日本」の IDF が 0 になり、「日本」だけが一致する文書のスコアが 0 になる。Lucene の IDF は 1 を足してから対数をとるので、必ず正になる。k1 の既定値（rank_bm25 は 1.5、Lucene は 1.2）や、単語の区切り方（unidic-lite と kuromoji に内蔵の辞書）も違う。
- kuromoji の検索モードは、長い複合語を、全体（関西国際空港）と部分（関西 / 国際 / 空港）の両方のトークンにするので、「空港」のような部分の語でも当たるようになる。通常モードでは複合語が 1 語のまま。標準のアナライザ (standard) は、漢字やひらがなを 1 文字ずつに分けてしまう（カタカナの続きは 1 語）。
- 品詞による除外 (`kuromoji_part_of_speech`) とストップワード (`ja_stop`) で、「で」「を」「とは」「です」のような機能語が消える（「BERT とは何ですか」→ bert, 何）。`kuromoji_baseform` は活用語を原形に戻す（買い → 買う）。その結果、「高さ」の「高」も「高い」になり、「日本で一番高い山」で東京スカイツリーの文書がヒットしている。原形化は検索漏れを減らすが、このような意図しない一致も増やす。
- 「日本一」は 1 語として解析されるので、「日本」とは一致しない。「大きい」と「最大」のような言い換えに弱いのも自前の BM25 と同じで、埋め込み検索との組み合わせ（この章の「キーワード検索と埋め込み検索を組み合わせる（ハイブリッド検索、RRF）」）が有効。
- `match` は既定で、クエリの語のどれかを含めばヒットする (OR)。`minimum_should_match` を 75% にすると、「日本で一番高い山」（4 語）では 3 語以上を含む富士山の文書だけになる。bool の `filter` はスコアを変えずに絞り込む。BM25 のパラメータは索引の `similarity` で指定でき、b=0 のフィールド (text.nolen) では、文書の長さ (dl) が 6、8、10 と違っても「日本」のスコアが同じになる。アナライザやマッピングは索引を作るときに決め、変えるときは文書を入れ直す。
- ここではセキュリティ機能（認証と HTTPS）を無効にしている。127.0.0.1 だけで待ち受ける学習用の設定であり、外から接続できるサーバーでは無効にしない。設定しないと、初回の起動時に証明書が作られて HTTPS と認証が必須になり、バックグラウンドで起動した場合は `bin/elasticsearch-reset-password` で elastic ユーザーのパスワードを設定する必要がある。
- 配布物 (tar.gz) のライセンスは Elastic License 2.0。ソースコードの大部分は AGPLv3、SSPL、Elastic License 2.0 から選べる（リポジトリの LICENSE.txt）。Apache License 2.0 だったころの Elasticsearch (7.10.2) のコードから派生した OpenSearch もあり、こちらは Apache License 2.0 で、analysis-kuromoji プラグインも用意されている。